# Tearing Simulation
This notebook details the functionality implemented to simulate the evolution and measurement of tearing modes.
1) Simulate an Initially Rotating Locked Mode (IRLM)
2) Measure the amplitude of toroidal mode numbers using the Low-N array
3) Model the impacts of noise or aliasing NTM's on the raw reconstructed signal, and plot the commonly-used RMS signal
3) To be added: Determine m/n components of IRLM using Mirnov array, simulate an Error Field Locked Mode (EFLM), and measure the amplitude of the EFLM using the Low-N array and/or flux loops.

First, we will use the tearing module to set up some IRLM's that are triggered and evolve using hard-coded values:

## Tearing Module

The tearing module is the thing which evolves a tearing mode over time. At the moment, it is only capable of evolving NTM's based on hard-coded values, but in the future we aim to add some amount of dynamics. Like all modules in POPSIM, the tearing module is composed of a state, parameters, outputs, and a config.

Tearing State:
A tearing mode is fully described by three values:
- `W`: The width of the magnetic island corresponding to the mode, in meters
- `F`: The frequency of toroidal rotation, in Hz
- `mode_phase`: The phase of toroidal rotation, in radians
Each of these quantities are 0 before the mode is triggered. Note that we define rotation and phase in the toroidal direction, but this is an arbitrary choice.

Tearing params:
The parameters determine when the mode should be triggered and how it should evolve. The most important are the `disruption_phase` and `rotation_phase` trajectories. 
- `disruption_phase` represents where in the disruption process the simulation is, and goes in the order `NONE`, `TQ`, `CQ`. 
- `rotation_phase` represents where in the mode evolution process the simulation is. 
    - For an IRLM, it goes in the order `NONE`, `SPAWN`, `ROTATING`, `DECELERATING`, and `LOCKED`
    - For an EFLM it may go `NONE`, `LOCKED`. The `SPAWN` state is instantaneous and exists for only one time step, it represents the initial kick which gets an IRLM to begin spinning.

Depending on the combinations of disruption phase and rotation phase, the mode will change in amplitude and rotation frequency differently according to hard coded values, though alternative values can passed in as parameters as well.

Tearing Config:
This describes the modes being simulated. The modes are defined by the poloidal and toroidal mode numbers $m$ and $n$, in a list like [(m1, n1), (m2, n2), ...]. The only modes presently implemented are (2, 1), (3, 1), and (3, 2).





In [ ]:
%load_ext autoreload
%autoreload 2

from popsim.modules.tearing import Tearing, generate_disruption_phase_trajectory, generate_tearing_phase_trajectory
from popsim.simulate import make_time_base

modes = [(2, 1), (3, 2)]
tearing_config = Tearing.Config(
    modes=modes,
)

tearing_initial_state = Tearing.State(
    W={mode: 0.0 for mode in modes}, F={mode: 0.0 for mode in modes}, mode_phase={mode: 0.0 for mode in modes}
)

dt = 1e-4 / 3  # s
time_base = make_time_base(t0=0.0, t1=4.0, dt=dt)

rot_dur = 1.0
locking_dur = 0.2
trigger_time = 2.0
disrupt_time = 3.5
dur_tq_to_spike = 1e-3

tearing_params = Tearing.Params(
    rot_dur=rot_dur,  # s
    locking_dur=locking_dur,  # s
    disruption_phase=generate_disruption_phase_trajectory(disrupt_time, dur_tq_to_spike, time_base, dt),
    tearing_phase=generate_tearing_phase_trajectory(trigger_time, rot_dur, locking_dur, time_base, dt),
)

tearing_module = Tearing(config=tearing_config)

## Low-N Array Module

The Low-N array is a set of magnetic probes that are used to measure the magnitude of toroidal magnetic perturbations. Since the goal is to measure the amplitude of potentially stationary modes which may be ~1 mT (compared to the 20 T toroidal field), each probe in the array is connected to another, and the difference between them is the measurement. At the moment, the design of this array is encoded in a text file `popsim/data/tearing/lown_design.txt`, but at some point in the future TODO(ZanderKeith) it should point to the device description (or at least something more permanent).

The Low-N array config also takes into account the frequency response of the probes, which is encoded in a text file `popsim/data/tearing/21_mode_resp_data.txt`.

Finally, there is the field `reconstructed_modes`, which are the modes the module will return measurements for. Note that the reconstructed mode numbers are independent from the tearing modes being simulated. The probe placements in this array were optimized to measure n=[1,2,3], but you could in principle resolve up to n=7 with the present design (16 probes, (16/2)-1 = 7). If you simulate higher n modes than what is being measured, the array will still return a measurement though it will be heavily skewed.

In [ ]:
from popsim.modules.magnetic_diagnostics import LowNArray, load_lown_config

probe_connections, func_Bp_per_A = load_lown_config()

lown_array_config = LowNArray.Config(func_Bp_per_A=func_Bp_per_A, probe_connections=probe_connections, reconstructed_modes=[1, 2, 3])

lown_array_module = LowNArray(config=lown_array_config)

## Tearing Simulation

Now that we have the modules set up, we can connect them together in the `TearingSim` class and get a measurement of our modes as they are evolved in time.

In [ ]:
from popsim.simulate import SimInput, simulate
from popsim.simulators.tearing_sim.model import TearingSim

sim_config = TearingSim.Config(
    tearing_module=tearing_module,
    lown_array_module=lown_array_module,
)

sim_initial_state = TearingSim.State(tearing_state=tearing_initial_state)

sim_params = TearingSim.Params(tearing_params=tearing_params)

tearing_sim = TearingSim(config=sim_config)

sim_input = SimInput(time=time_base, initial_state=sim_initial_state, params=sim_params)

sim_xarray = simulate(tearing_sim, sim_inputs=sim_input)

In [ ]:
import holoviews as hv

hv.extension("matplotlib")

print(sim_xarray)

def tearing_sim_plots(sim_xarray, modes: list[tuple[int, int]]):
    width_plots = []
    frequency_plots = []
    phase_plots = []
    for mode in modes:
        width_plots.append(hv.Scatter((sim_xarray.time, sim_xarray[f"state.tearing_state.W.{mode}"]), label=f"{mode}"))
        frequency_plots.append(hv.Scatter((sim_xarray.time, sim_xarray[f"state.tearing_state.F.{mode}"]), label=f"{mode}"))
        phase_plots.append(hv.Scatter((sim_xarray.time, sim_xarray[f"state.tearing_state.mode_phase.{mode}"]), label=f"{mode}"))

    reconstructed_n1 = hv.Scatter((sim_xarray.time, sim_xarray["output.locals.reconstructed_magnitudes.1"]), label="n=1")
    reconstructed_n2 = hv.Scatter((sim_xarray.time, sim_xarray["output.locals.reconstructed_magnitudes.2"]), label="n=2")
    reconstructed_n3 = hv.Scatter((sim_xarray.time, sim_xarray["output.locals.reconstructed_magnitudes.3"]), label="n=3")

    width_plot = hv.Overlay(width_plots).opts(title="Tearing mode widths", xlabel="Time [s]", ylabel="Island Width [m]")
    frequency_plot = hv.Overlay(frequency_plots).opts(title="Tearing mode frequencies", xlabel="Time [s]", ylabel="Frequency [Hz]")
    phase_plot = hv.Overlay(phase_plots).opts(title="Tearing mode phases", xlabel="Time [s]", ylabel="Phase [rad]")
    reconstructed_plot = (reconstructed_n1 * reconstructed_n2 * reconstructed_n3).opts(title="Reconstructed mode amplitudes", xlabel="Time [s]", ylabel="Amplitude [T]")

    return width_plot, frequency_plot, phase_plot, reconstructed_plot

width_plot, frequency_plot, phase_plot, reconstructed_plot = tearing_sim_plots(sim_xarray, modes)
layout = (width_plot + frequency_plot + phase_plot + reconstructed_plot).opts(shared_axes=False)
layout

## Noise and Aliasing

The reconstructions above are almost perfect. Too perfect. In reality, the signals will be noisy which can make the reconstructed magnitude vary widely over time. To smooth out these effects, we implement the RMS signal.

In [ ]:
modes = [(2, 1), (3, 1)]
tearing_config = Tearing.Config(
    modes=modes,
)

tearing_initial_state = Tearing.State(
    W={mode: 0.0 for mode in modes}, F={mode: 0.0 for mode in modes}, mode_phase={mode: 0.0 for mode in modes}
)

dt = 1e-4 / 3  # s
time_base = make_time_base(t0=0.0, t1=4.0, dt=dt)

rot_dur = 1.0
locking_dur = 0.2
trigger_time = 2.0
disrupt_time = 3.5
dur_tq_to_spike = 1e-3

tearing_params = Tearing.Params(
    rot_dur=rot_dur,  # s
    locking_dur=locking_dur,  # s
    disruption_phase=generate_disruption_phase_trajectory(disrupt_time, dur_tq_to_spike, time_base, dt),
    tearing_phase=generate_tearing_phase_trajectory(trigger_time, rot_dur, locking_dur, time_base, dt),
    initial_rot_freq={(2, 1): 7e3, (3, 1): 10e3}, # Just an illustration of aliasing
)

tearing_module = Tearing(config=tearing_config)

sim_config = TearingSim.Config(
    tearing_module=tearing_module,
    lown_array_module=lown_array_module,
)

sim_initial_state = TearingSim.State(tearing_state=tearing_initial_state)

sim_params = TearingSim.Params(tearing_params=tearing_params)

tearing_sim = TearingSim(config=sim_config)

sim_input = SimInput(time=time_base, initial_state=sim_initial_state, params=sim_params)

sim_xarray = simulate(tearing_sim, sim_inputs=sim_input)

width_plot, frequency_plot, phase_plot, reconstructed_plot = tearing_sim_plots(sim_xarray, modes)

# Plot the tearing mode widths and frequencies next to eachother, but keep the y axes separate
layout = (width_plot + frequency_plot + phase_plot + reconstructed_plot).opts(shared_axes=False)
layout